# Stage-4 Coverage Diagnostic

Hypothesis: the 32 fixed Stage-0 indices produce fewer GT-positive Stage-4 superpoint pairs than natural voxelization,
which would explain PIR plateauing at 0.1–0.2 for 20+ epochs.

**Measurement:** for 20 val samples, count how many Stage-4 ref superpoints have at least one Stage-4 src
superpoint within `ground_truth_matching_radius=0.05m` after applying the GT transform.

In [ ]:
import sys, os, pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import KDTree

EXPERIMENT_DIR = os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(EXPERIMENT_DIR, '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
if EXPERIMENT_DIR not in sys.path:
    sys.path.insert(0, EXPERIMENT_DIR)

from geotransformer.modules.ops import grid_subsample
from geotransformer.modules.ops.transformation import apply_transform
from geotransformer.utils.pointcloud import get_transform_from_rotation_translation

## 1. Reconstruct avg_ref (z=0) from PCA basis

In [ ]:
DATASET_ROOT = os.path.abspath(os.path.join(EXPERIMENT_DIR, '..', '..', 'data', 'faces'))

pca = torch.load(os.path.join(EXPERIMENT_DIR, 'pca_basis_all.pth'), weights_only=False)
print('PCA keys:', list(pca.keys()))
for k, v in pca.items():
    print(f'  {k}: {v.shape}')

pca_mean = pca['mean']          # [32, 2100]
patch_indices = pca['patch_indices']  # [32, 700]

num_patches = patch_indices.shape[0]
k_neighbors = patch_indices.shape[1]

flat_indices = patch_indices.view(-1).long()
num_global_verts = flat_indices.max().item() + 1
print(f'\nnum_global_verts: {num_global_verts}')

# Reconstruct avg_ref using last-write-wins (same logic as generate_reference_geometry with z=0)
reconstructed_patches = pca_mean.view(num_patches, k_neighbors, 3)  # [32, 700, 3]
flat_points = reconstructed_patches.contiguous().view(-1, 3)         # [22400, 3]

patch_scores = torch.arange(num_patches, dtype=torch.long)
flat_scores = patch_scores.unsqueeze(1).expand(-1, k_neighbors).reshape(-1)

max_scores = torch.full((num_global_verts,), -1, dtype=torch.long)
max_scores.scatter_reduce_(0, flat_indices, flat_scores, reduce='amax', include_self=False)

is_max_score = flat_scores == max_scores[flat_indices]
valid_flat_positions = torch.arange(flat_indices.size(0))[is_max_score]
valid_global_indices = flat_indices[is_max_score]

best_idx_per_global = torch.zeros(num_global_verts, dtype=torch.long)
best_idx_per_global.scatter_(0, valid_global_indices, valid_flat_positions)

avg_ref = torch.zeros((num_global_verts, 3), dtype=torch.float32)
has_points = torch.zeros(num_global_verts, dtype=torch.bool)
has_points[valid_global_indices] = True
avg_ref[has_points] = flat_points[best_idx_per_global[has_points]]

print(f'avg_ref shape: {avg_ref.shape}')
print(f'Zero-point vertices (uncovered): {(~has_points).sum().item()}')

## 2. Natural voxelization helper

In [ ]:
def get_stage4_natural(ref_pts, src_pts, init_vs=0.025, num_stages=4):
    """Returns (stage4_ref, stage4_src) via standard grid_subsample, no fixed-indices override."""
    ref_n0 = ref_pts.shape[0]
    src_n0 = src_pts.shape[0]
    pts = torch.cat([ref_pts, src_pts], dim=0).cpu()
    lens = torch.tensor([ref_n0, src_n0], dtype=torch.long)
    vs = init_vs
    for i in range(num_stages):
        if i > 0:
            pts, lens = grid_subsample(pts, lens, voxel_size=vs)
        vs *= 2
    ref_n = lens[0].item()
    return pts[:ref_n], pts[ref_n:]

## 3. Fixed Stage-4 ref points

In [ ]:
fixed_indices = torch.tensor([
    75, 411, 2699, 911, 8594, 3380, 6731, 9710, 9633, 119,
    3441, 6319, 9541, 8732, 6162, 3774, 8296, 3151, 10,
    7720, 6858, 7409, 7531, 3504, 6937, 4189, 8891, 3721,
    9241, 2213, 1765, 7547
], dtype=torch.long)

fixed_stage4_ref = avg_ref[fixed_indices]  # [32, 3]
print(f'Fixed Stage-4 ref: {fixed_stage4_ref.shape}')
print(f'Spatial spread (std): {fixed_stage4_ref.std(dim=0).numpy()}')

## 4. Visualize spatial coverage: fixed vs natural

In [ ]:
# Get natural Stage-4 ref by running avg_ref through the pipeline with a dummy src
dummy_src = avg_ref[:100]  # tiny dummy; only ref Stage-4 matters here
natural_stage4_ref, _ = get_stage4_natural(avg_ref, dummy_src)
print(f'Natural Stage-4 ref: {natural_stage4_ref.shape[0]} points')
print(f'Fixed Stage-4 ref:   {fixed_stage4_ref.shape[0]} points')

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ref_np = avg_ref.numpy()
ax1.scatter(ref_np[::10, 0], ref_np[::10, 1], ref_np[::10, 2], c='lightgrey', s=1, alpha=0.3)
ax1.scatter(fixed_stage4_ref[:, 0], fixed_stage4_ref[:, 1], fixed_stage4_ref[:, 2],
            c='red', s=80, marker='x', linewidths=2, label=f'Fixed ({len(fixed_stage4_ref)})')
ax1.set_title('Fixed 32 indices (Stage-4 ref)')
ax1.legend()
ax1.view_init(elev=20, azim=0)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(ref_np[::10, 0], ref_np[::10, 1], ref_np[::10, 2], c='lightgrey', s=1, alpha=0.3)
ax2.scatter(natural_stage4_ref[:, 0], natural_stage4_ref[:, 1], natural_stage4_ref[:, 2],
            c='blue', s=40, marker='o', label=f'Natural ({len(natural_stage4_ref)})')
ax2.set_title('Natural voxelization (Stage-4 ref)')
ax2.legend()
ax2.view_init(elev=20, azim=0)

plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, 'debug_stage4_coverage_viz.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved viz')

## 5. Count GT positive pairs over 20 val samples

In [ ]:
with open(os.path.join(DATASET_ROOT, 'metadata', 'val.pkl'), 'rb') as f:
    metadata = pickle.load(f)

# Val scale = 1.0 (default, no augmentation)
# If you trained with a fixed val scale != 1.0, change this
VAL_SCALE = 1.0
MATCHING_RADIUS = 0.05
N_SAMPLES = min(20, len(metadata))

fixed_counts, natural_counts, natural_totals = [], [], []
natural_src_totals = []

for i, meta in enumerate(metadata[:N_SAMPLES]):
    src = torch.load(os.path.join(DATASET_ROOT, 'data', meta['pcd1']), weights_only=False)
    if isinstance(src, np.ndarray):
        src = torch.from_numpy(src.astype(np.float32))
    src = src.float()
    src_scaled = src * VAL_SCALE

    # Get natural Stage-4 for this sample
    stage4_ref_nat, stage4_src_nat = get_stage4_natural(avg_ref.float(), src_scaled)

    # Apply GT transform to bring src Stage-4 into canonical (ref) frame
    transform = get_transform_from_rotation_translation(meta['rotation'], meta['translation'])
    transform_t = torch.from_numpy(transform.astype(np.float32))

    # src is in scaled space; transform maps unscaled src → canonical
    # apply to unscaled src Stage-4 for correct canonical-frame positions
    if VAL_SCALE != 1.0:
        stage4_src_unscaled = stage4_src_nat / VAL_SCALE
    else:
        stage4_src_unscaled = stage4_src_nat
    stage4_src_canon = apply_transform(stage4_src_unscaled, transform_t).numpy()

    if len(stage4_src_canon) == 0:
        print(f'Sample {i}: empty src Stage-4, skipping')
        continue

    tree = KDTree(stage4_src_canon)

    d_fixed, _ = tree.query(fixed_stage4_ref.numpy(), k=1)
    d_nat, _ = tree.query(stage4_ref_nat.numpy(), k=1)

    n_fixed = int((d_fixed < MATCHING_RADIUS).sum())
    n_nat = int((d_nat < MATCHING_RADIUS).sum())

    fixed_counts.append(n_fixed)
    natural_counts.append(n_nat)
    natural_totals.append(len(stage4_ref_nat))
    natural_src_totals.append(len(stage4_src_canon))

    print(f'Sample {i:2d}: fixed={n_fixed:2d}/32 | natural={n_nat:2d}/{len(stage4_ref_nat):2d} | src_pts={len(stage4_src_canon):2d}')

print()
print(f'=== SUMMARY ===')
print(f'Fixed:   avg {np.mean(fixed_counts):.1f} ± {np.std(fixed_counts):.1f} / 32 positive pairs')
print(f'Natural: avg {np.mean(natural_counts):.1f} ± {np.std(natural_counts):.1f} / {np.mean(natural_totals):.0f} ref pts positive')
print(f'Natural src Stage-4: avg {np.mean(natural_src_totals):.1f} pts')

ratio_fixed = np.mean(fixed_counts) / 32
ratio_nat = np.array(natural_counts) / np.array(natural_totals)
print(f'\nFixed positive rate:   {ratio_fixed:.2%}')
print(f'Natural positive rate: {np.mean(ratio_nat):.2%}')

## 6. Summary bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(fixed_counts))
axes[0].bar(x - 0.2, fixed_counts, 0.4, label='Fixed (32 idx)', color='tomato')
axes[0].bar(x + 0.2, natural_counts, 0.4, label='Natural', color='steelblue')
axes[0].axhline(np.mean(fixed_counts), color='red', linestyle='--', alpha=0.7, label=f'Fixed mean={np.mean(fixed_counts):.1f}')
axes[0].axhline(np.mean(natural_counts), color='blue', linestyle='--', alpha=0.7, label=f'Natural mean={np.mean(natural_counts):.1f}')
axes[0].set_xlabel('Val sample index')
axes[0].set_ylabel('GT positive Stage-4 pairs')
axes[0].set_title('GT Positive Pairs: Fixed vs Natural (absolute)')
axes[0].legend()

fixed_rates = [c/32 for c in fixed_counts]
natural_rates = [c/t for c, t in zip(natural_counts, natural_totals)]
axes[1].bar(x - 0.2, fixed_rates, 0.4, label='Fixed', color='tomato')
axes[1].bar(x + 0.2, natural_rates, 0.4, label='Natural', color='steelblue')
axes[1].axhline(np.mean(fixed_rates), color='red', linestyle='--', alpha=0.7)
axes[1].axhline(np.mean(natural_rates), color='blue', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Val sample index')
axes[1].set_ylabel('Fraction of ref superpoints with a match')
axes[1].set_title('GT Positive Rate: Fixed vs Natural')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, 'debug_stage4_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

# Decision
print('\n=== DECISION ===')
if np.mean(natural_counts) >= 2 * np.mean(fixed_counts):
    print('NATURAL gives 2x+ more positive pairs → REMOVE fixed_indices block from geotransformer/utils/data.py:34-44')
elif np.mean(natural_counts) > np.mean(fixed_counts) * 1.3:
    print('NATURAL gives ~30-100% more positive pairs → removing fixed indices likely helps, but benefit is moderate')
else:
    print('Counts are similar → fixed indices are NOT the bottleneck; investigate elsewhere')